# A gentle introduction to Feast
**Prepare once. Reuse in two models and an application.**

We will use one housing file and just four features: area, bedrooms, area per bedroom, and house age.

Select your existing `.venv` Python kernel and run the cells in order.

In [9]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name != "feature_store_feast":
    ROOT = ROOT / "feature_store_feast"
assert (ROOT / "get_data.py").exists(), "Start in the project root or feature_store_feast folder."
sys.path.insert(0, str(ROOT))

print("Python:", sys.executable)

Python: C:\Users\Ash\Desktop\educational\play_ML\.venv\Scripts\python.exe


## 1. Get the raw data
`get_data.py` generates raw observations of 100 houses over 24 months. It creates no engineered feature columns.

Inspect the table: it has area, bedrooms, year built, timestamps, and a synthetic target price. `house_id` is a lookup key and `sale_price` is the target; neither is a model feature.

In [11]:
from get_data import get_data
from common import FEATURE_COLUMNS, FEATURE_REFS, store

print("FEATURE_COLUMNS:", FEATURE_COLUMNS)
print("FEATURE_REFS:",  FEATURE_REFS)

raw_data = get_data()
display(raw_data.head())

print("Raw columns:", raw_data.columns.tolist())

FEATURE_COLUMNS: ['area_sqft', 'bedrooms', 'area_per_bedroom', 'house_age_years']
FEATURE_REFS: ['house_features:area_sqft', 'house_features:bedrooms', 'house_features:area_per_bedroom', 'house_features:house_age_years']
Step 1: saved 2400 raw observations to data/raw_housing.parquet


,house_id,area_sqft,bedrooms,year_built,event_timestamp,sale_price
0,1,905,4,1994,2024-11-01 00:00:00+00:00,235559.80
1,2,2480,1,2016,2024-11-01 00:00:00+00:00,495970.49
2,3,2205,4,1999,2024-11-01 00:00:00+00:00,462313.65
3,4,1709,1,2007,2024-11-01 00:00:00+00:00,336263.46
4,5,1695,4,1998,2024-11-01 00:00:00+00:00,354353.17


Raw columns: ['house_id', 'area_sqft', 'bedrooms', 'year_built', 'event_timestamp', 'sale_price']


## 2. Transform the raw data into features
Now inspect the separate `transform_features.py` file. Its `transform_features(raw_data)` function adds:
- `area_per_bedroom = area_sqft / bedrooms`
- `house_age_years = snapshot year - year_built`

`prepare_features()` reads the raw file, calls that function, and saves the result. We display the before-and-after tables separately, so the pipeline stages are visible.

Feast reads only the prepared `housing.parquet` file. Both models select their inputs from this one prepared feature view.

In [3]:
from transform_features import prepare_features

data = prepare_features()
display(data.head())

print("Our four features:", FEATURE_COLUMNS)
print("Raw data still has no engineered columns:", raw_data.columns.tolist())

Step 2: saved prepared features to data/housing.parquet


,house_id,area_sqft,bedrooms,year_built,event_timestamp,sale_price,area_per_bedroom,house_age_years
0,1,905,4,1994,2024-11-01 00:00:00+00:00,235559.80,226.25,30
1,2,2480,1,2016,2024-11-01 00:00:00+00:00,495970.49,2480.00,8
2,3,2205,4,1999,2024-11-01 00:00:00+00:00,462313.65,551.25,25
3,4,1709,1,2007,2024-11-01 00:00:00+00:00,336263.46,1709.00,17
4,5,1695,4,1998,2024-11-01 00:00:00+00:00,354353.17,423.75,26


Our four features: ['area_sqft', 'bedrooms', 'area_per_bedroom', 'house_age_years']
Raw data still has no engineered columns: ['house_id', 'area_sqft', 'bedrooms', 'year_built', 'event_timestamp', 'sale_price']


## 3. Tell Feast where the features are
Read `feature_repo/definitions.py`:
- **Entity:** a house, identified by `house_id`.
- **Source:** the one `housing.parquet` file.
- **Feature view:** a named group of four prepared features.

`feast apply` registers these definitions. Feast does not do the feature engineering for us.

In [4]:
subprocess.run(
    [sys.executable, "-c", "from feast.cli.cli import cli; cli()", "apply"],
    cwd=ROOT / "feature_repo", check=True,
)

fs = store()

print("Registered views:", [view.name for view in fs.list_feature_views()])

Registered views: ['house_features']


## 4. Ask for features for a particular date
House age depends on the year. A training example from last year should use last year's age.

`get_historical_features()` retrieves the latest snapshot at or before the date we ask for. This is a **point-in-time lookup**. 

Here we query the **same house at two dates and see its age change**. The view allows snapshots up to 40 days old, enough for monthly observations.

In [5]:
house_history = data[data.house_id == 1].sort_values("event_timestamp")
requests = house_history.iloc[[0, -1]][["house_id", "event_timestamp"]]

features_at_two_dates = fs.get_historical_features(
    entity_df=requests,
    features=FEATURE_REFS,
).to_df()

display(features_at_two_dates.sort_values("event_timestamp"))

,house_id,event_timestamp,area_sqft,bedrooms,area_per_bedroom,house_age_years
0,1,2024-11-01 00:00:00+00:00,905,4,226.25,30
1,1,2026-10-01 00:00:00+00:00,905,4,226.25,32


## 5. Train different models with different feature subsets
The store has four features, but a model can ask for a smaller subset.

| Model | Features |
| --- | --- |
| Linear Regression | area and bedrooms (two features) |
| Random Forest | area, bedrooms, area per bedroom, and house age (four features) |

`MODEL_FEATURES` in `common.py` specifies the subsets. `train_models.py` asks Feast for each subset separately, then trains the corresponding model. Both use the same feature view and dates, and neither repeats feature engineering.

Earlier months train the models; the last four months test them. MAE means average absolute price error; lower is better. Different algorithms and inputs can give different errors.

In [6]:
from common import MODEL_FEATURES

# Show how a model asks Feast for only two of the four available features.
small_subset = MODEL_FEATURES["linear"]
small_training_data = fs.get_historical_features(
    entity_df=data[["house_id", "event_timestamp", "sale_price"]].head(5),
    features=[f"house_features:{column}" for column in small_subset],
).to_df()
display(small_training_data)

from train_models import train
training_data = train()

,house_id,event_timestamp,sale_price,area_sqft,bedrooms
0,1,2024-11-01 00:00:00+00:00,235559.80,905,4
1,2,2024-11-01 00:00:00+00:00,495970.49,2480,1
2,3,2024-11-01 00:00:00+00:00,462313.65,2205,4
3,4,2024-11-01 00:00:00+00:00,336263.46,1709,1
4,5,2024-11-01 00:00:00+00:00,354353.17,1695,4


linear uses 2 features: ['area_sqft', 'bedrooms']
  Test MAE = $12,301.10
forest uses 4 features: ['area_sqft', 'bedrooms', 'area_per_bedroom', 'house_age_years']
  Test MAE = $8,736.75
Different feature subsets, one shared Feast feature view.


## 6. Make the latest features available for prediction
The **offline store** is our Parquet history, used for training.

The **online store** is a small SQLite database with the latest values for each house. **Materialization** means copying the latest features into this database. It does not recompute them.

In [7]:
from materialize import materialize
materialize()

latest = fs.get_online_features(
    features=FEATURE_REFS,
    entity_rows=[{"house_id": 1}],
).to_df()

display(latest)

Materializing 1 feature views from 2024-11-01 00:00:00+00:00 to 2026-10-01 00:00:01+00:00 into the sqlite online store.

house_features:


,house_id,area_per_bedroom,area_sqft,house_age_years,bedrooms
0,1,226.25,905,32,4


## 7. Reuse each model's subset in predictions
`predict.py` needs only a house ID. It retrieves two features for Linear Regression and four for Random Forest, using the same per-model lists as training. No feature engineering is repeated.

To experiment, change `MODEL_FEATURES` in `common.py`, restart the kernel, and rerun the notebook so the models are retrained.

In [8]:
from predict import predict

predictions = predict(house_id=1)

predictions

linear: features retrieved from Feast:
 area_sqft  bedrooms
       905         4
linear prediction: $228,642.94
forest: features retrieved from Feast:
 area_sqft  bedrooms  area_per_bedroom  house_age_years
       905         4            226.25               32
forest prediction: $223,077.70


{'linear': 228642.94, 'forest': 223077.7}

## Optional: use the same code in FastAPI
After this lesson, activate `.venv`, open a terminal in `feature_store_feast`, and run:

```powershell
python -m uvicorn app:app --host 127.0.0.1 --port 8000
```

Open http://127.0.0.1:8000/predict/1. The application calls the same `predict()` function.

The path we demonstrated is:

**Get raw data → engineer features → register with Feast → train two models → predict.**

Feast does not train models or refresh the data automatically. Rerun the demo to refresh this dataset. See `readme.md` for commands and definitions.